In [ ]:
# Top-k vs Top-p (Nucleus) Sampling

## Goal

Compare two common sampling constraints:

- **Top-k**: keep the k most likely tokens
- **Top-p** (nucleus): keep the smallest set of tokens whose cumulative probability >= p

We keep everything else constant:
- same model
- same prompt
- same temperature
- same seed

We observe differences in:
- diversity
- stability
- constraint-following

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

In [ ]:
prompt = """
Explain photosynthesis.

Return exactly 5 bullet points.
Each bullet must:
- Start with "- "
- Contain at most 12 words
- Use simple vocabulary
""".strip()

In [ ]:
def generate(prompt, seed=0, temperature=1.0, top_k=None, top_p=None):
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        max_new_tokens=120,
        do_sample=True,
        temperature=temperature,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    if top_k is not None:
        gen_kwargs["top_k"] = top_k
    if top_p is not None:
        gen_kwargs["top_p"] = top_p

    out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
print("=== Baseline: temperature=1.0 (no top-k, no top-p) ===\n")
print(generate(prompt, seed=42, temperature=1.0))

In [ ]:
print("\n=== Top-k only: top_k=20 ===\n")
print(generate(prompt, seed=42, temperature=1.0, top_k=20))

In [ ]:
print("\n=== Top-p only: top_p=0.9 ===\n")
print(generate(prompt, seed=42, temperature=1.0, top_p=0.9))

In [ ]:
print("\n=== Compare top-k values ===\n")
for k in [10, 20, 50]:
    print(f"\n--- top_k={k} ---")
    print(generate(prompt, seed=42, temperature=1.0, top_k=k))

In [ ]:
print("\n=== Compare top-p values ===\n")
for p in [0.8, 0.9, 0.95]:
    print(f"\n--- top_p={p} ---")
    print(generate(prompt, seed=42, temperature=1.0, top_p=p))

In [ ]:
## Observations

### Top-k
- Keeps a fixed number of candidate tokens.
- Smaller k reduces diversity strongly.
- Larger k allows more exploration.

### Top-p
- Keeps a variable number of tokens based on probability mass.
- Adaptive: if the distribution is sharp, it keeps fewer tokens.
- Often produces smoother diversity control than top-k.

## Key Insight

Top-k and top-p both restrict randomness,
but they do so in different ways:

- Top-k = fixed candidate count
- Top-p = fixed probability mass

This means their impact depends on how "peaked" the token distribution is.